<a href="https://colab.research.google.com/github/sandeshdevre25/data-engineering/blob/main/DE_prac_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import pandas as pd
import sqlite3
import json

# Connect to database
conn = sqlite3.connect("etl_database.db")

# 1. Extract CSV, clean, transform and load
df = pd.DataFrame({
    "ID": [1, 2, 3, 4],
    "Name": [" Rahul ", "Priya", "", "Amit"],
    "Age": [20, 21, 19, 22],
    "Marks": [85, 90, 75, 110]
})

df["Name"] = df["Name"].str.strip()
clean = df[
    (df["Name"] != "") &
    (df["Age"].between(1, 100)) &
    (df["Marks"].between(0, 100))
]
clean.to_sql("Students", conn, if_exists="replace", index=False)
print("1. Cleaned CSV Data:\n", clean)

# 2. Combine multiple CSV files
df1 = pd.DataFrame({"ID": [1, 2], "Name": ["Rahul", "Priya"]})
df2 = pd.DataFrame({"ID": [3, 4], "Name": ["Amit", "Neha"]})

combined = pd.concat([df1, df2], ignore_index=True)
combined.to_sql("Combined_Students", conn, if_exists="replace", index=False)
print("\n2. Combined Data:\n", combined)

# 3. Extract JSON, transform and load
json_data = '''
[
 {"id": 1, "name": "rahul", "age": 20},
 {"id": 2, "name": "priya", "age": 21}
]'''

records = json.loads(json_data)
json_df = pd.json_normalize(records)
json_df["name"] = json_df["name"].str.upper()
json_df.to_sql("JSON_Students", conn, if_exists="replace", index=False)
print("\n3. JSON Data:\n", json_df)

# 4. Identify invalid records
data = pd.DataFrame({
    "ID": [1, 2, 3],
    "Name": ["Rahul", "", "Priya"],
    "Age": [20, 21, -5],
    "Marks": [85, 90, 80]
})

invalid = data[
    (data["Name"].str.strip() == "") |
    (~data["Age"].between(1, 100)) |
    (~data["Marks"].between(0, 100))
]
valid = data.drop(invalid.index)
print("\n4. Invalid Records:\n", invalid)
print("Valid Records:\n", valid)

# 5. Validate before loading
valid.to_sql("Validated_Students", conn, if_exists="replace", index=False)
print("\n5. Validated records saved to database.")

# 6. Incremental loading: insert only new IDs
initial = pd.DataFrame({
    "ID": [1, 2, 3],
    "Name": ["Rahul", "Priya", "Amit"]
})
initial.to_sql("Employees", conn, if_exists="replace", index=False)

new_data = pd.DataFrame({
    "ID": [3, 4, 5],
    "Name": ["Amit", "Neha", "Rohan"]
})

existing_ids = pd.read_sql_query(
    "SELECT ID FROM Employees", conn
)["ID"]

incremental = new_data[~new_data["ID"].isin(existing_ids)]
incremental.to_sql("Employees", conn, if_exists="append", index=False)

print("\n6. Incremental Data Loaded:\n",
      pd.read_sql_query("SELECT * FROM Employees", conn))

conn.close()
print("\nAll 6 ETL practicals completed successfully!")

1. Cleaned CSV Data:
    ID   Name  Age  Marks
0   1  Rahul   20     85
1   2  Priya   21     90

2. Combined Data:
    ID   Name
0   1  Rahul
1   2  Priya
2   3   Amit
3   4   Neha

3. JSON Data:
    id   name  age
0   1  RAHUL   20
1   2  PRIYA   21

4. Invalid Records:
    ID   Name  Age  Marks
1   2          21     90
2   3  Priya   -5     80
Valid Records:
    ID   Name  Age  Marks
0   1  Rahul   20     85

5. Validated records saved to database.

6. Incremental Data Loaded:
    ID   Name
0   1  Rahul
1   2  Priya
2   3   Amit
3   4   Neha
4   5  Rohan

All 6 ETL practicals completed successfully!
